In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# STEP 2: Import Libraries
import os, re, random
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer
from sklearn.utils import resample
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import seaborn as sns

# STEP 3: Define Paths
data_root = "path"
train_dir = os.path.join(data_root, "path")
test_dir = os.path.join(data_root, "path")
metadata_dir = os.path.join(data_root, "path")
metadata_path = os.path.join(metadata_dir, "path")
test_metadata_path = os.path.join(metadata_dir, "path")
synthetic_dir = "path"

# STEP 4: Extract Upper Triangle
def extract_upper_triangle(file_path):
    matrix = pd.read_csv(file_path, sep='\t', header=None).values
    upper_tri_indices = np.triu_indices_from(matrix, k=1)
    return matrix[upper_tri_indices]

# STEP 5: Load Real Training Data
train_vectors, train_ids = [], []
print("🔄 Processing training files...")
for filename in tqdm(os.listdir(train_dir)):
    if filename.endswith(".tsv"):
        participant_id = filename.split("_")[0].replace("sub-", "").replace(".tsv", "")
        file_path = os.path.join(train_dir, filename)
        vector = extract_upper_triangle(file_path)
        train_vectors.append(vector)
        train_ids.append(participant_id)

X_train_raw = pd.DataFrame(train_vectors)
X_train_raw["participant_id"] = train_ids

metadata = pd.read_csv(metadata_path)
X_train_raw["participant_id"] = X_train_raw["participant_id"].astype(str).str.upper()
metadata["participant_id"] = metadata["participant_id"].astype(str).str.upper()

# STEP 5b: Incorporate Synthetic Data with Sampling + Midpoint Age
synthetic_vectors, synthetic_ids, synthetic_ages = [], [], []
synthetic_files = [f for f in os.listdir(synthetic_dir) if f.endswith(".csv")]
random.seed(42)
sampled_files = random.sample(synthetic_files, int(0.1 * len(synthetic_files)))  # 10% sample

print("🔄 Incorporating sampled synthetic data...")
for filename in tqdm(sampled_files):
    match = re.search(r'(\d+)p(\d+)_(\d+)p(\d+)', filename)
    if match:
        low = float(f"{int(match.group(1))}.{int(match.group(2)):02d}")
        high = float(f"{int(match.group(3))}.{int(match.group(4)):02d}")
        age_mid = (low + high) / 2
    else:
        continue
    file_path = os.path.join(synthetic_dir, filename)
    matrix = pd.read_csv(file_path, header=None).values
    vector = matrix[np.triu_indices_from(matrix, k=1)]
    synthetic_vectors.append(vector)
    synthetic_ids.append(f"SYNTH_{filename}")
    synthetic_ages.append(age_mid)

synthetic_df = pd.DataFrame(synthetic_vectors)
synthetic_df["participant_id"] = synthetic_ids
synthetic_df["age"] = synthetic_ages

real_df = pd.merge(X_train_raw, metadata[["participant_id", "age"]], on="participant_id")
train_df = pd.concat([real_df, synthetic_df], ignore_index=True)
print(f"✅ Total dataset shape: {train_df.shape}")

# ✅ DIAGNOSTIC PLOT
plt.figure(figsize=(10, 5))
sns.histplot(train_df["age"], bins=30)
plt.title("Age distribution (real + synthetic data)")
plt.show()
print(train_df["age"].value_counts().sort_index())

# STEP 6: Smart Resampling
min_samples_per_age = 50
balanced = []
for age, group in train_df.groupby("age"):
    if len(group) < min_samples_per_age:
        resampled = resample(group, replace=True, n_samples=min_samples_per_age, random_state=42)
        balanced.append(resampled)
    else:
        balanced.append(group)
df_resampled = pd.concat(balanced)
print(f"✅ Resampled dataset shape: {df_resampled.shape}")

plt.figure(figsize=(10, 5))
sns.histplot(df_resampled["age"], bins=30)
plt.title("Age distribution after smart resampling")
plt.show()

# STEP 7: Preprocess (PCA + Metadata Safe)
brain_cols = [col for col in df_resampled.columns if isinstance(col, int) or (isinstance(col, str) and col.isdigit())]
meta_cols = [col for col in df_resampled.columns if col not in brain_cols + ["participant_id", "age"]]

scaler = StandardScaler()
brain_scaled = scaler.fit_transform(df_resampled[brain_cols])
pca = PCA(n_components=0.95)
brain_pca = pca.fit_transform(brain_scaled)

if meta_cols:
    meta_encoded = pd.get_dummies(df_resampled[meta_cols]).reindex(columns=None, fill_value=0)
    X = np.concatenate([brain_pca, meta_encoded.values], axis=1)
else:
    X = brain_pca

y = df_resampled["age"]
qt = QuantileTransformer(output_distribution='normal')
y_transformed = qt.fit_transform(y.values.reshape(-1, 1)).ravel()
X = SimpleImputer(strategy='mean').fit_transform(X)

# STEP 7a: Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X, y)
print("✅ Ridge RMSE:", -cross_val_score(ridge, X, y, cv=5, scoring='neg_root_mean_squared_error').mean())

# STEP 8: Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_split=2, max_features=20, random_state=42)
rf.fit(X, y)
print("✅ RF RMSE:", -cross_val_score(rf, X, y, cv=5, scoring='neg_root_mean_squared_error').mean())

# STEP 8b: XGBoost
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8,
                   subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb.fit(X, y)
print("✅ XGB RMSE:", -cross_val_score(xgb, X, y, cv=5, scoring='neg_root_mean_squared_error').mean())

# STEP 9: Process Test Data
test_vectors, test_ids = [], []
print("🔄 Processing test files...")
for filename in tqdm(os.listdir(test_dir)):
    if filename.endswith(".tsv"):
        pid = filename.split("_")[0].replace("sub-", "").replace(".tsv", "")
        vector = extract_upper_triangle(os.path.join(test_dir, filename))
        test_vectors.append(vector)
        test_ids.append(pid)

X_test_raw = pd.DataFrame(test_vectors)
X_test_raw["participant_id"] = test_ids
test_meta = pd.read_csv(test_metadata_path)
test_meta["participant_id"] = test_meta["participant_id"].astype(str).str.upper()
X_test_raw["participant_id"] = X_test_raw["participant_id"].astype(str).str.upper()
test_df = pd.merge(X_test_raw, test_meta, on="participant_id")

test_brain_scaled = scaler.transform(test_df[brain_cols])
test_brain_pca = pca.transform(test_brain_scaled)

if meta_cols:
    test_meta_encoded = pd.get_dummies(test_df[meta_cols])
    test_meta_encoded = test_meta_encoded.reindex(columns=meta_encoded.columns, fill_value=0)
    X_test_final = np.concatenate([test_brain_pca, test_meta_encoded.values], axis=1)
else:
    X_test_final = test_brain_pca

# STEP 10: Predict
predictions = xgb.predict(X_test_final)
predictions = np.clip(predictions, 8, 21)

submission = pd.DataFrame({"participant_id": test_df["participant_id"], "age": predictions})
submission.to_csv(os.path.join(data_root, "final_submission.csv"), index=False)
print("✅ Submission saved.")

# STEP 11: Visualize
sns.histplot(predictions, bins=20)
plt.title("Predicted Age Distribution")
plt.show()
plt.show()


In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# STEP 2: Import Libraries
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer
from sklearn.utils import resample
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import seaborn as sns

# STEP 3: Define Paths
data_root = "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)"
train_dir = os.path.join(data_root, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/train_tsv/train_tsv")
test_dir = os.path.join(data_root, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/test_tsv/test_tsv")
metadata_dir = os.path.join(data_root, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/metadata")
metadata_path = os.path.join(metadata_dir, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/metadata/training_metadata.csv")
test_metadata_path = os.path.join(metadata_dir, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/metadata/test_metadata.csv")
#synthetic_dir = "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/Necessary Generated Data"

# STEP 4: Extract Upper Triangle
def extract_upper_triangle(file_path):
    matrix = pd.read_csv(file_path, sep='\t', header=None).values
    upper_tri_indices = np.triu_indices_from(matrix, k=1)
    return matrix[upper_tri_indices]

# STEP 5: Load Real Training Data
train_vectors, train_ids = [], []
print("🔄 Processing training files...")
for filename in tqdm(os.listdir(train_dir)):
    if filename.endswith(".tsv"):
        participant_id = filename.split("_")[0].replace("sub-", "").replace(".tsv", "")
        file_path = os.path.join(train_dir, filename)
        vector = extract_upper_triangle(file_path)
        train_vectors.append(vector)
        train_ids.append(participant_id)

X_train_raw = pd.DataFrame(train_vectors)
X_train_raw["participant_id"] = train_ids

metadata = pd.read_csv(metadata_path)
X_train_raw["participant_id"] = X_train_raw["participant_id"].astype(str).str.upper()
metadata["participant_id"] = metadata["participant_id"].astype(str).str.upper()

train_df = pd.merge(X_train_raw, metadata[["participant_id", "age"]], on="participant_id")
print(f"✅ Loaded real training data: {train_df.shape}")

# ✅ DIAGNOSTIC PLOT
plt.figure(figsize=(10, 5))
sns.histplot(train_df["age"], bins=30)
plt.title("Age distribution before resampling")
plt.show()
print(train_df["age"].value_counts().sort_index())

# STEP 6: Smart Resampling (only real data)
min_samples_per_age = 50   # 👈 adjust as needed
balanced = []
for age, group in train_df.groupby("age"):
    if len(group) < min_samples_per_age:
        resampled = resample(group, replace=True, n_samples=min_samples_per_age, random_state=42)
        balanced.append(resampled)
    else:
        balanced.append(group)
df_resampled = pd.concat(balanced)
print(f"✅ Resampled dataset shape: {df_resampled.shape}")

plt.figure(figsize=(10, 5))
sns.histplot(df_resampled["age"], bins=30)
plt.title("Age distribution after resampling")
plt.show()

# STEP 7: Preprocess (PCA + Metadata Safe)
brain_cols = [col for col in df_resampled.columns if isinstance(col, int) or (isinstance(col, str) and col.isdigit())]
meta_cols = [col for col in df_resampled.columns if col not in brain_cols + ["participant_id", "age"]]

scaler = StandardScaler()
brain_scaled = scaler.fit_transform(df_resampled[brain_cols])
pca = PCA(n_components=0.95)
brain_pca = pca.fit_transform(brain_scaled)

if meta_cols:
    meta_encoded = pd.get_dummies(df_resampled[meta_cols]).reindex(columns=None, fill_value=0)
    X = np.concatenate([brain_pca, meta_encoded.values], axis=1)
else:
    X = brain_pca

y = df_resampled["age"]
qt = QuantileTransformer(output_distribution='normal')
y_transformed = qt.fit_transform(y.values.reshape(-1, 1)).ravel()
X = SimpleImputer(strategy='mean').fit_transform(X)

# STEP 7a: Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X, y)
print("✅ Ridge RMSE:", -cross_val_score(ridge, X, y, cv=5, scoring='neg_root_mean_squared_error').mean())

# STEP 8: Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_split=2, max_features=20, random_state=42)
rf.fit(X, y)
print("✅ RF RMSE:", -cross_val_score(rf, X, y, cv=5, scoring='neg_root_mean_squared_error').mean())

# STEP 8b: XGBoost
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8,
                   subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb.fit(X, y)
print("✅ XGB RMSE:", -cross_val_score(xgb, X, y, cv=5, scoring='neg_root_mean_squared_error').mean())

# STEP 9: Process Test Data
test_vectors, test_ids = [], []
print("🔄 Processing test files...")
for filename in tqdm(os.listdir(test_dir)):
    if filename.endswith(".tsv"):
        pid = filename.split("_")[0].replace("sub-", "").replace(".tsv", "")
        vector = extract_upper_triangle(os.path.join(test_dir, filename))
        test_vectors.append(vector)
        test_ids.append(pid)

X_test_raw = pd.DataFrame(test_vectors)
X_test_raw["participant_id"] = test_ids
test_meta = pd.read_csv(test_metadata_path)
test_meta["participant_id"] = test_meta["participant_id"].astype(str).str.upper()
X_test_raw["participant_id"] = X_test_raw["participant_id"].astype(str).str.upper()
test_df = pd.merge(X_test_raw, test_meta, on="participant_id")

test_brain_scaled = scaler.transform(test_df[brain_cols])
test_brain_pca = pca.transform(test_brain_scaled)

if meta_cols:
    test_meta_encoded = pd.get_dummies(test_df[meta_cols])
    test_meta_encoded = test_meta_encoded.reindex(columns=meta_encoded.columns, fill_value=0)
    X_test_final = np.concatenate([test_brain_pca, test_meta_encoded.values], axis=1)
else:
    X_test_final = test_brain_pca

# STEP 10: Predict
predictions = xgb.predict(X_test_final)
predictions = np.clip(predictions, 8, 21)

submission = pd.DataFrame({"participant_id": test_df["participant_id"], "age": predictions})
submission.to_csv(os.path.join(data_root, "final_submission_real_only.csv"), index=False)
print("✅ Submission saved.")

# STEP 11: Visualize Predictions
sns.histplot(predictions, bins=20)
plt.title("Predicted Age Distribution (Real Data Only)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Diagnostic: plot age distribution after adding synthetic
plt.figure(figsize=(10, 5))
sns.histplot(train_df["age"], bins=30, kde=False)
plt.title("Age distribution (real + synthetic data)")
plt.xlabel("Age")
plt.ylabel("Count")
plt.grid(True)
plt.show()

# Also print a table of age counts
print(train_df["age"].value_counts().sort_index())

In [ ]:
from sklearn.utils import resample

min_samples_per_age = 50  # 👈 adjust this number as you want

balanced_frames = []
for age, group in train_df.groupby("age"):
    if len(group) < min_samples_per_age:
        # Oversample to reach min_samples_per_age
        group_resampled = resample(group, replace=True, n_samples=min_samples_per_age, random_state=42)
        balanced_frames.append(group_resampled)
    else:
        # Keep as is
        balanced_frames.append(group)

df_resampled = pd.concat(balanced_frames)
print(f"✅ Resampled dataset shape: {df_resampled.shape}")

# Optional: visualize resampled distribution
plt.figure(figsize=(10, 5))
sns.histplot(df_resampled["age"], bins=30, kde=False)
plt.title("Age distribution after smart resampling")
plt.xlabel("Age")
plt.ylabel("Count")
plt.grid(True)
plt.show()